# Лабораторная работа №3
## Train/test split, кросс-валидация и подбор K для k-NN

Методичка: [LAB_TMO__KNN](https://github.com/ugapanyuk/courses_current/wiki/LAB_TMO__KNN).

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42

wine = load_wine(as_frame=True)
X = wine.data
y = wine.target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
])
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)
print("accuracy:", accuracy_score(y_test, pred))
print("macro-f1:", f1_score(y_test, pred, average="macro"))

In [ ]:
pipe = Pipeline([( "scaler", StandardScaler()), ("knn", KNeighborsClassifier())])
param_grid = {
    "knn__n_neighbors": list(range(1, 22, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}
cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv1, scoring="f1_macro", n_jobs=-1)
grid.fit(X_train, y_train)
print("best params:", grid.best_params_)
pred_g = grid.predict(X_test)
print("grid macro-f1:", f1_score(y_test, pred_g, average="macro"))

In [ ]:
param_dist = {
    "knn__n_neighbors": list(range(1, 31)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}
cv2 = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
rand = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=40, cv=cv2, scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=-1)
rand.fit(X_train, y_train)
print("best params:", rand.best_params_)
pred_r = rand.predict(X_test)
print("rand macro-f1:", f1_score(y_test, pred_r, average="macro"))